## Pipeline RAG

Este projeto implementa um sistema de recuperação de informação baseado em **busca semântica**, permitindo encontrar trechos relevantes em documentos textuais a partir do significado da consulta, em vez de simples correspondência de palavras-chave.

O sistema é projetado para:
- Processar documentos HTML (ex: livros do Project Gutenberg)
- Gerar representações vetoriais (embeddings)
- Executar buscas eficientes por similaridade semântica

### Arquitetura do Sistema

O pipeline é dividido em cinco etapas principais:

1. **Ingestão de dados**
2. **Pré-processamento**
3. **Segmentação (chunking)**
4. **Geração de embeddings**
5. **Busca semântica**

### Decisões de Design

#### 1. Uso de Embeddings

Foi utilizado o modelo `sentence-transformers/all-MiniLM-L6-v2` por ter baixo custo computacional e boa performance em tarefas de similaridade semântica.

#### 2. Estratégia de Chunking

**Parâmetros:**

- `chunk_size = 800`  
- `chunk_overlap = 100`  

Evita perda de contexto semântico entre chunks e o Overlap reduz problemas em fronteiras de texto.

#### 3. Normalização de Vetores

Todos os vetores são normalizados antes da comparação para garantir o uso correto da similaridade de cosseno e evitar distorções por diferença de magnitude.

#### 4. Armazenamento

**Formato:**

- `pickle`

Escolhido pela simplicidade e rapidez, sendo adequado para datasets pequenos e médios.

#### 5. Componentes do Sistema

##### 5.1. Ingestão e Limpeza

Responsável por:

- Remover HTML irrelevante (`script`, `style`)  
- Extrair texto puro  
- Normalizar espaçamento  
- Remover metadados do Project Gutenberg  

##### 5.2. Segmentação (Chunking)

Divide documentos longos em partes menores.

Estrutura de cada chunk:

```json
{
  "id": "doc_chunk_0",
  "text": "conteudo...",
  "document": "nome_doc"
}
```

##### 5.3. Geração de Embeddings

Transforma textos em vetores numéricos

##### 5.4. Mecanismo de Busca

Processo:

- Geração do embedding da query
- Cálculo da similaridade de cosseno
- Ordenação por relevância
- Aplicação de threshold
- Retorno dos top-K resultados

#### Imports

In [ ]:
import numpy as np
import pickle
import re
import os
from bs4 import BeautifulSoup

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

####  Configuração e inicialização do Modelo de Embeddings
Como modelo de embedding foi utilizado um da biblioteca `sentence-transformers`.

In [ ]:
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBEDDINGS_FILE = 'embeddings.pickle'

embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

#### Carregamento e Limpeza
Esta função:
- Lê arquivos HTML
- Remove scripts e estilos
- Extrai texto puro
- Remove ruídos (como cabeçalhos do Project Gutenberg)

In [ ]:
def load_and_clean_docs(folder_path: str) -> list:
    documents = []

    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)

        with open(file_path, 'r', encoding='utf-8') as f:
            html_source = f.read()

        soup = BeautifulSoup(html_source, "html.parser")

        for tag in soup(["script", "style"]):
            tag.decompose()

        text = soup.get_text(separator="\n")

        text = re.sub(r'\n+', '\n', text)
        text = re.sub(r'[ \t]+', ' ', text)

        start_match = re.search(r"\*\*\* START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*\*\*\*", text, re.IGNORECASE)

        if start_match:
            text = text[start_match.end():]

        end_match = re.search(r"\*\*\* END OF (THE|THIS) PROJECT GUTENBERG EBOOK.*\*\*\*", text, re.IGNORECASE)

        if end_match:
            text = text[:end_match.start()]

        text = text.strip()
        document_name = os.path.splitext(file_name)[0]

        documents.append({
            'document': document_name,
            'text': text
        })

    return documents

#### Divisão em Chunks
Divide os textos em partes menores para melhorar a busca semântica.

In [ ]:
def chunking(documents: list, chunk_size=800, chunk_overlap=100) -> list:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

    chunks = []

    for doc in documents:
        split_texts = text_splitter.split_text(doc['text'])

        for i, chunk in enumerate(split_texts):
            chunks.append({
                'id': f"{doc['document']}_chunk_{i}",
                'text': chunk,
                'document': doc['document']
            })

    return chunks

#### Geração de Embeddings
Transforma cada chunk em um vetor numérico que representa seu significado.

In [ ]:
def make_embeddings(chunks: list, output_file=EMBEDDINGS_FILE) -> list:
    texts = [c['text'] for c in chunks]

    vectors = embedding_model.embed_documents(texts)

    for i, vec in enumerate(vectors):
        chunks[i]['embedding'] = vec

    with open(output_file, 'wb') as f:
        pickle.dump(chunks, f)

    normalized_vectors = normalize(vectors)

    return chunks, normalized_vectors

#### Carregamento dos Embeddings Salvos

In [ ]:
def load_embeddings(file: str) -> list:
    with open(file, 'rb') as f:
        chunks = pickle.load(f)

    vectors = [c['embedding'] for c in chunks]
    normalized_vectors = normalize(vectors)

    return chunks, normalized_vectors

#### Busca Semântica

Realiza busca por similaridade de cosseno entre query e chunks.

In [ ]:
def search(embeddings:list, chunk_vectors:np.ndarray, query:str | list, top_k:int=10, score_threshold:float=0.3) -> list:
    queries = [query] if isinstance(query, str) else query

    query_vectors = normalize([embedding_model.embed_query(q) for q in queries])

    all_results = []

    for query_idx, query_vec in enumerate(query_vectors):
        similarity_scores = cosine_similarity([query_vec], chunk_vectors)[0]

        ranked_indices = np.argsort(-similarity_scores)

        results = []
        seen_ids = set()

        for i in ranked_indices:
            if embeddings[i]['id'] in seen_ids:
                continue

            score = float(similarity_scores[i])

            if score < score_threshold:
                continue

            seen_ids.add(embeddings[i]['id'])

            results.append({
                'query': queries[query_idx],
                'text': embeddings[i]['text'][:300] + "...",
                'document': embeddings[i]['document'],
                'score': round(score, 4)
            })

            if len(results) == top_k:
                break

        all_results.append(results)

    return all_results

#### Exibição dos Resultados

In [ ]:
def print_results(results: list):
    for query_results in results:
        if not query_results:
            continue
        print(f'Query: {query_results[0]['query']}')

        for i, r in enumerate(query_results, 1):
            print(f'Top {i}')
            print(f"Documento: {r['document']}")
            print(f"Score: {r['score']:.4f}")
            print(f"Texto:\n{r['text']}")

#### Execução do Pipeline

In [ ]:
if not os.path.exists(EMBEDDINGS_FILE):
    print('Gerando embeddings...')
    docs = load_and_clean_docs(folder_path='docs/')
    chunks = chunking(documents=docs)
    embeddings, vectors = make_embeddings(chunks=chunks)
else:
    print('Carregando embeddings')
    embeddings, vectors = (load_embeddings('embeddings.pickle'))

##### Exemplo com uma lista de perguntas

In [ ]:
queries = [
    "What is the central reason characters create false identities in The Importance of Being Earnest?,"
    "How does Jekyll’s transformation into Hyde represent the conflict between morality and human instinct?",
    "What events illustrate the absurd logic of Wonderland in Alice’s Adventures in Wonderland?",
    "Besides the family feud, what factors lead to the tragedy in Romeo and Juliet?"
]

In [ ]:
results = search(embeddings, vectors, queries)

print_results(results)

##### Exemplo com apenas uma pergunta

In [ ]:
query = "In The Adventures of Sherlock Holmes, what is Sherlock Holmes’s address?"

In [ ]:
result = search(embeddings, vectors, query)

print_results(result)